Durante las últimas semanas SuperData ha seguido registrando ventas. El departamento de TI nos ha enviado una nueva extracción de datos (clientes_nuevo.csv, productos_nuevo.csv y ventas_nuevo.csv).

Antes de incorporarlos al sistema, debemos comprobar si presentan problemas de calidad y, en caso necesario, aplicar las transformaciones oportunas.

In [536]:
import polars as pl

# Clientes:

In [537]:
# Leer el csv de clientes
clientes_nuevo = pl.read_csv("../data_raw/clientes_nuevo.csv")

In [538]:
def valores_minimos_inchoherentes(df,columna,valor):
    return df.filter(
        pl.col(columna) < valor
    )
def valores_maximos_inchoherentes(df,columna,valor):
    return df.filter(
        pl.col(columna) > valor
    )
def id_duplicados(df,columna):
    return df.filter(
        pl.col(columna).is_duplicated()
    )
def textos_unicos(df,columna):
    return df.select(
        pl.col(columna).unique()
    )
def eliminar_registro_por_id(df, columna_id, id_eliminar):
    return df.filter(
        pl.col(columna_id) != id_eliminar
    )
def columna_nula(df,columna):
    return df.filter(
        pl.col(columna).is_null()
    )
def campo_minuscula(df,columna):
    return df.with_columns(
        pl.col(columna).str.to_lowercase()
    )

In [539]:
clientes_nuevo.head()

fila,id_cliente,nombre,apellidos,sexo,edad,ciudad,provincia,codigo_postal,fecha_alta,socio,tipo_cliente
i64,i64,str,str,str,i64,str,str,i64,str,bool,str
0,1,"""Álvaro""","""Castro Navarro""","""Mujer""",24,"""Valencia""","""Valencia""",35314,"""2023-04-25""",false,"""Normal"""
1,42,"""María""","""Castro Navarro""","""Mujer""",56,"""Bilbao""","""Bizkaia""",39225,"""2022-11-18""",true,"""Normal"""
2,47,"""Miguel""","""Castro Torres""","""Mujer""",64,"""Vigo""","""Pontevedra""",13158,"""2023-05-23""",true,"""Normal"""
3,66,"""Sergio""","""Castro Romero""","""Mujer""",27,"""Santander""","""Cantabria""",20567,"""2023-10-22""",true,"""Normal"""
4,68,"""Claudia""","""Fernández Ortega""","""Mujer""",63,"""A Coruña""","""A Coruña""",47525,"""2024-11-03""",false,"""Ocasional"""


In [540]:
clientes_nuevo.describe()

statistic,fila,id_cliente,nombre,apellidos,sexo,edad,ciudad,provincia,codigo_postal,fecha_alta,socio,tipo_cliente
str,f64,f64,str,str,str,f64,str,str,f64,str,f64,str
"""count""",51.0,51.0,"""51""","""51""","""51""",51.0,"""51""","""51""",51.0,"""51""",51.0,"""51"""
"""null_count""",0.0,0.0,"""0""","""0""","""0""",0.0,"""0""","""0""",0.0,"""0""",0.0,"""0"""
"""mean""",24.019608,269.705882,null,null,null,45.901961,null,null,24919.901961,null,0.588235,null
"""std""",14.833058,139.419266,null,null,null,17.868693,null,null,17662.096074,null,null,null
"""min""",0.0,1.0,"""Alejandro""","""Castro García""","""Hombre""",18.0,"""A Coruña""","""A Coruña""",123.0,"""2022-04-03""",0.0,"""Frecuente"""
"""25%""",12.0,161.0,null,null,null,32.0,null,null,7771.0,null,null,null
"""50%""",24.0,287.0,null,null,null,45.0,null,null,24569.0,null,null,null
"""75%""",37.0,390.0,null,null,null,63.0,null,null,42692.0,null,null,null
"""max""",49.0,493.0,"""Álvaro""","""Álvarez Pérez""","""Mujer""",80.0,"""bilbao""","""Zaragoza""",52656.0,"""2024-12-17""",1.0,"""Ocasional"""


In [541]:
clientes_nuevo.dtypes

[Int64,
 Int64,
 String,
 String,
 String,
 Int64,
 String,
 String,
 Int64,
 String,
 Boolean,
 String]

In [542]:
# Valores nulos:
clientes_nuevo.null_count()

fila,id_cliente,nombre,apellidos,sexo,edad,ciudad,provincia,codigo_postal,fecha_alta,socio,tipo_cliente
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0


En este caso no hay nulos. Pero si hubiera:

| Estrategia                                | Cuándo utilizarla                                            |
| ----------------------------------------- | ------------------------------------------------------------ |
| Eliminar el registro (`drop_nulls`)       | Cuando son pocos registros o la información es irrelevante.  |
| Sustituir por un valor fijo (`fill_null`) | Por ejemplo, descuento = 0 o stock = 0.                      |
| Sustituir por la media                    | Variables numéricas como precio o edad.                      |
| Sustituir por la mediana                  | Cuando existen valores extremos que pueden sesgar la media.  |
| Sustituir por la moda                     | Variables categóricas como ciudad o método de pago.          |
| Buscar el dato en otra fuente             | Si existe otra base de datos o sistema con esa información.  |
| Mantener el valor nulo                    | Cuando no es posible inferir el dato sin introducir errores. |


In [543]:
# Valores incosistentes
clientes_nuevo["ciudad"].value_counts().sort("count", descending=True)

ciudad,count
str,u32
"""Vigo""",6
"""Salamanca""",5
"""Logroño""",4
"""Bilbao""",4
"""Valencia""",3
…,…
"""Burgos""",1
"""León""",1
"""MADRID""",1


In [544]:
display(textos_unicos(clientes_nuevo,"sexo"))
display(textos_unicos(clientes_nuevo,"tipo_cliente"))

sexo
str
"""Hombre"""
"""Mujer"""


tipo_cliente
str
"""Frecuente"""
"""Ocasional"""
"""Normal"""


In [545]:
clientes_nuevo = clientes_nuevo.with_columns(
    pl.col("ciudad")
      .str.to_titlecase()
)

clientes_nuevo["ciudad"].value_counts()

ciudad,count
str,u32
"""Oviedo""",3
"""Barcelona""",2
"""Zaragoza""",2
"""Málaga""",1
"""Bilbao""",5
…,…
"""Gijón""",2
"""Alicante""",2
"""Salamanca""",5


In [546]:
clientes_nuevo = eliminar_registro_por_id(clientes_nuevo, "id_cliente", 1)

In [547]:
clientes_nuevo.filter(
    pl.col("id_cliente").is_duplicated()
)

fila,id_cliente,nombre,apellidos,sexo,edad,ciudad,provincia,codigo_postal,fecha_alta,socio,tipo_cliente
i64,i64,str,str,str,i64,str,str,i64,str,bool,str


In [548]:
for col in clientes_nuevo.columns:
    nulos = columna_nula(clientes_nuevo, col).height
    print(f"{col}: {nulos} valores nulos")
    

fila: 0 valores nulos
id_cliente: 0 valores nulos
nombre: 0 valores nulos
apellidos: 0 valores nulos
sexo: 0 valores nulos
edad: 0 valores nulos
ciudad: 0 valores nulos
provincia: 0 valores nulos
codigo_postal: 0 valores nulos
fecha_alta: 0 valores nulos
socio: 0 valores nulos
tipo_cliente: 0 valores nulos


In [549]:
# Duplicados
print("Registros:", clientes_nuevo.height)

print("Registros únicos:", clientes_nuevo.unique().height)

Registros: 49
Registros únicos: 49


In [550]:
clientes_nuevo = clientes_nuevo.unique()

clientes_nuevo.shape

(49, 12)

In [551]:
# En vez de eliminarlos, podemos ponerlos a nulo para que luego podamos imputarlos
clientes_nuevo = clientes_nuevo.with_columns(
    pl.when(
        pl.col("codigo_postal").cast(pl.Utf8).str.len_chars() != 5
    )
    .then(pl.col("codigo_postal"))
    .otherwise(None)
)
clientes_nuevo.filter(
  pl.col("codigo_postal").is_null()
)

fila,id_cliente,nombre,apellidos,sexo,edad,ciudad,provincia,codigo_postal,fecha_alta,socio,tipo_cliente
i64,i64,str,str,str,i64,str,str,i64,str,bool,str
49,493,"""Alejandro""","""Romero Gómez""","""Hombre""",72,"""Santander""","""Cantabria""",null,"""2023-04-09""",false,"""Normal"""
35,368,"""Sergio""","""Jiménez Díaz""","""Mujer""",34,"""Burgos""","""Burgos""",null,"""2024-07-12""",true,"""Ocasional"""
4,68,"""Claudia""","""Fernández Ortega""","""Mujer""",63,"""A Coruña""","""A Coruña""",null,"""2024-11-03""",false,"""Ocasional"""
33,344,"""Laura""","""Sánchez Pérez""","""Mujer""",58,"""Madrid""","""Madrid""",null,"""2024-04-03""",false,"""Normal"""
7,98,"""Sara""","""García Pérez""","""Mujer""",23,"""Bilbao""","""Bizkaia""",null,"""2023-09-04""",false,"""Normal"""
…,…,…,…,…,…,…,…,…,…,…,…
26,297,"""Julia""","""Romero García""","""Hombre""",35,"""Oviedo""","""Asturias""",null,"""2022-04-03""",false,"""Normal"""
6,93,"""Alejandro""","""Navarro Martín""","""Hombre""",28,"""Alicante""","""Alicante""",null,"""2023-10-30""",true,"""Normal"""
15,226,"""Miguel""","""Navarro Sánchez""","""Mujer""",70,"""Zaragoza""","""Zaragoza""",null,"""2022-09-24""",false,"""Normal"""


In [552]:
# Valores incorrectos
#clientes_nuevo.filter(
#    pl.col("codigo_postal")
#      .cast(pl.Utf8)
#      .str.len_chars() != 5
#)

In [553]:
display(valores_minimos_inchoherentes(clientes_nuevo,"edad",0))
display(valores_maximos_inchoherentes(clientes_nuevo,"edad",110))
display(id_duplicados(clientes_nuevo,"id_cliente"))


fila,id_cliente,nombre,apellidos,sexo,edad,ciudad,provincia,codigo_postal,fecha_alta,socio,tipo_cliente
i64,i64,str,str,str,i64,str,str,i64,str,bool,str


fila,id_cliente,nombre,apellidos,sexo,edad,ciudad,provincia,codigo_postal,fecha_alta,socio,tipo_cliente
i64,i64,str,str,str,i64,str,str,i64,str,bool,str


fila,id_cliente,nombre,apellidos,sexo,edad,ciudad,provincia,codigo_postal,fecha_alta,socio,tipo_cliente
i64,i64,str,str,str,i64,str,str,i64,str,bool,str


¿Qué hacemos con estos valores?

No hay una única respuesta correcta.

Opciones:

Los codigo postales españoles tienen 5 digitos, al ser los clientes de provincias españolas ese dato es erroneo, como no creo que sea un dato tan importante, los datos incorrectos los paso a nulos

- eliminarlos
- marcarlos como inválidos
- buscar el CP correcto en otra fuente

# Productos:

In [554]:
productos_nuevo = pl.read_csv("../data_raw/productos_nuevo.csv") 
productos_nuevo.head()

id_producto,nombre,categoria,marca,precio,coste
i64,str,str,str,f64,f64
11,"""Fanta""","""Bebidas""","""Nestlé""",1.56,0.87
13,"""Cerdo""","""Carnes""","""Bimbo""",18.52,9.2
23,"""Lechuga""","""Verduras""","""Hero""",null,4.79
24,"""Snacks perro""","""Mascotas""","""SuperData""",11.74,6.5
25,"""Pan de semillas""","""Panadería""","""Danone""",9.49,5.41


In [555]:
productos_nuevo.describe()

statistic,id_producto,nombre,categoria,marca,precio,coste
str,f64,str,str,str,f64,f64
"""count""",31.0,"""31""","""31""","""30""",30.0,31.0
"""null_count""",0.0,"""0""","""0""","""1""",1.0,0.0
"""mean""",81.225806,null,null,null,6.656,3.989355
"""std""",44.334118,null,null,null,5.441722,2.866452
"""min""",11.0,""" Vino""","""BEBIDAS""","""Bimbo""",-5.5,0.38
"""25%""",43.0,null,null,null,2.76,1.81
"""50%""",90.0,null,null,null,5.71,3.69
"""75%""",123.0,null,null,null,10.06,5.73
"""max""",148.0,"""Zanahoria""","""bebidas""","""SuperData""",18.52,12.65


In [556]:
productos_nuevo.dtypes

[Int64, String, String, String, Float64, Float64]

In [557]:
productos_nuevo.null_count()

id_producto,nombre,categoria,marca,precio,coste
u32,u32,u32,u32,u32,u32
0,0,0,1,1,0


In [558]:
display(valores_minimos_inchoherentes(productos_nuevo,"precio",0))
display(valores_minimos_inchoherentes(productos_nuevo,"coste",0))
display(id_duplicados(productos_nuevo,"id_producto"))
display(textos_unicos(productos_nuevo,"categoria"))


id_producto,nombre,categoria,marca,precio,coste
i64,str,str,str,f64,f64
95,"""Molde""","""Panadería""","""Danone""",-5.5,2.66


id_producto,nombre,categoria,marca,precio,coste
i64,str,str,str,f64,f64


id_producto,nombre,categoria,marca,precio,coste
i64,str,str,str,f64,f64
11,"""Fanta""","""Bebidas""","""Nestlé""",1.56,0.87
11,"""Fanta""","""Bebidas""","""Nestlé""",1.56,0.87


categoria
str
"""Bebidas"""
"""Lácteos"""
"""bebidas"""
"""Panadería"""
"""Higiene"""
…
"""Pescados"""
"""Carnes"""
"""Frutas"""


In [559]:
productos_nuevo = eliminar_registro_por_id(productos_nuevo,"id_producto", 11)
display(id_duplicados(productos_nuevo,"id_producto"))

id_producto,nombre,categoria,marca,precio,coste
i64,str,str,str,f64,f64


In [560]:
#Podriamos volver a llamar a la funcion eliminar, pero en este caso tiene sentido cambiar signo
#productos_nuevo = eliminar_registro_por_id(productos_nuevo,"id_producto", 95)
productos_nuevo = productos_nuevo.with_columns(
    pl.when(
        pl.col("id_producto") == 95
    ).then(pl.col("precio").abs())
    .otherwise(pl.col("precio"))
    .alias("precio")
)


In [561]:
productos_nuevo.filter(
    pl.col("nombre") == "Molde"
)

id_producto,nombre,categoria,marca,precio,coste
i64,str,str,str,f64,f64
95,"""Molde""","""Panadería""","""Danone""",5.5,2.66


In [562]:
productos_nuevo = campo_minuscula(productos_nuevo,"categoria")
display(textos_unicos(productos_nuevo,"categoria"))

categoria
str
"""higiene"""
"""panadería"""
"""desayuno"""
"""congelados"""
"""bebidas"""
…
"""verduras"""
"""pescados"""
"""mascotas"""


In [563]:
for col in productos_nuevo.columns:
    nulos = columna_nula(productos_nuevo, col).height
    print(f"{col}: {nulos} valores nulos")
    

id_producto: 0 valores nulos
nombre: 0 valores nulos
categoria: 0 valores nulos
marca: 1 valores nulos
precio: 1 valores nulos
coste: 0 valores nulos


In [564]:
productos_nuevo.filter(
    pl.col("marca").is_null()
)

id_producto,nombre,categoria,marca,precio,coste
i64,str,str,str,f64,f64
38,"""Snacks perro""","""mascotas""",null,10.06,5.73


In [565]:
productos_nuevo.filter(
    pl.col("precio").is_null()
)

id_producto,nombre,categoria,marca,precio,coste
i64,str,str,str,f64,f64
23,"""Lechuga""","""verduras""","""Hero""",null,4.79


In [566]:
productos_nuevo = eliminar_registro_por_id(productos_nuevo,"id_producto",23)

In [567]:
print("Registros:", productos_nuevo.height)
print("Registros únicos:", productos_nuevo.unique().height)

Registros: 28
Registros únicos: 28


Se ha decidido eliminar el registro cuyo id = 11, ya que habia dos registros con el mismo id, teniendo que ser esto completamente imposible.
He investigado, ya que tenemos un registro, que es un molde con valor negativo, lo cual es imposible, he buscado si teniamos otro registro con el mismo nombre, pero de otra categoria, para la posibilidad de estimar un precio pero al no haberlo se ha tenido que eliminar
Por otra parte teniamos registros en categoria escrito de diferentes maneras, como puede resultar confuso lo hemos transformado todo a minusculas

# Ventas:

In [568]:
ventas_nuevo = pl.read_csv("../data_raw/ventas_nuevo.csv") 
ventas_nuevo.head()

id_venta,fecha,id_cliente,id_producto,cantidad,metodo_pago,descuento
i64,str,i64,i64,i64,str,i64
19,"""2025-03-09 19:00:00""",156,117,2,"""Tarjeta""",0
237,"""2025-02-01 10:00:00""",441,2,2,"""Efectivo""",10
255,"""2025-10-31 10:00:00""",76,73,4,"""Tarjeta""",10
354,"""2025-03-23 19:00:00""",483,104,3,"""Tarjeta""",null
404,"""2025-04-12 16:00:00""",370,42,1,"""Tarjeta""",20


In [569]:
ventas_nuevo.dtypes

[Int64, String, Int64, Int64, Int64, String, Int64]

In [570]:
ventas_nuevo.describe()

statistic,id_venta,fecha,id_cliente,id_producto,cantidad,metodo_pago,descuento
str,f64,str,f64,f64,f64,str,f64
"""count""",151.0,"""151""",151.0,151.0,151.0,"""150""",150.0
"""null_count""",0.0,"""0""",0.0,0.0,0.0,"""1""",1.0
"""mean""",5259.384106,null,261.251656,77.13245,2.072848,null,5.266667
"""std""",2871.891007,null,135.362536,45.06864,1.108148,null,13.194297
"""min""",19.0,"""2025-01-02 16:00:00""",7.0,1.0,-2.0,"""Bizum""",0.0
"""25%""",2816.0,null,156.0,42.0,1.0,null,0.0
"""50%""",5501.0,null,264.0,76.0,2.0,null,0.0
"""75%""",7578.0,null,381.0,116.0,3.0,null,10.0
"""max""",10000.0,"""2025-12-31 11:00:00""",493.0,150.0,5.0,"""tarjeta""",150.0


In [571]:
for col in ventas_nuevo.columns:
    nulos = columna_nula(ventas_nuevo, col).height
    print(f"{col}: {nulos} valores nulos")
    

id_venta: 0 valores nulos
fecha: 0 valores nulos
id_cliente: 0 valores nulos
id_producto: 0 valores nulos
cantidad: 0 valores nulos
metodo_pago: 1 valores nulos
descuento: 1 valores nulos


In [572]:
textos_unicos(ventas_nuevo,"metodo_pago")

metodo_pago
str
"""tarjeta"""
"""Efectivo"""
"""bizum"""
"""Bizum"""
"""TARJETA"""
null
"""Cheque regalo"""
"""Tarjeta"""


In [573]:
ventas_nuevo = campo_minuscula(ventas_nuevo,"metodo_pago")
display(textos_unicos(ventas_nuevo,"metodo_pago"))

metodo_pago
str
"""tarjeta"""
null
"""efectivo"""
"""bizum"""
"""cheque regalo"""


In [574]:
display(columna_nula(ventas_nuevo,"metodo_pago"))

id_venta,fecha,id_cliente,id_producto,cantidad,metodo_pago,descuento
i64,str,i64,i64,i64,str,i64
519,"""2025-06-29 15:00:00""",101,3,1,null,0


In [575]:
# No se como de importante es que sea nulo el metodo de pago
#ventas_nuevo = eliminar_registro_por_id(ventas_nuevo,"id_venta", 519)

In [576]:
display(columna_nula(ventas_nuevo,"descuento"))

id_venta,fecha,id_cliente,id_producto,cantidad,metodo_pago,descuento
i64,str,i64,i64,i64,str,i64
354,"""2025-03-23 19:00:00""",483,104,3,"""tarjeta""",null


In [577]:
ventas_nuevo = ventas_nuevo.with_columns(
    pl.when(pl.col("descuento").is_null())
    .then(0)
    .otherwise(pl.col("descuento"))
    .alias("descuento")
)
ventas_nuevo.filter(
    pl.col("id_venta") == 354
)

id_venta,fecha,id_cliente,id_producto,cantidad,metodo_pago,descuento
i64,str,i64,i64,i64,str,i64
354,"""2025-03-23 19:00:00""",483,104,3,"""tarjeta""",0


In [579]:
ventas_detalle = (
    ventas_nuevo
    .join(
        clientes_nuevo.select([
            "id_cliente",
            "nombre",
            "apellidos"
        ]),
        on="id_cliente",
        how="left"
    )
    .join(
        productos_nuevo.select([
            "id_producto"
        ]),
        on="id_producto",
        how="left"
    )
)

ventas_detalle

id_venta,fecha,id_cliente,id_producto,cantidad,metodo_pago,descuento,nombre,apellidos
i64,str,i64,i64,i64,str,i64,str,str
19,"""2025-03-09 19:00:00""",156,117,2,"""tarjeta""",0,null,null
237,"""2025-02-01 10:00:00""",441,2,2,"""efectivo""",10,null,null
255,"""2025-10-31 10:00:00""",76,73,4,"""tarjeta""",10,null,null
354,"""2025-03-23 19:00:00""",483,104,3,"""tarjeta""",0,null,null
404,"""2025-04-12 16:00:00""",370,42,1,"""tarjeta""",20,null,null
…,…,…,…,…,…,…,…,…
9842,"""2025-02-15 22:00:00""",264,78,3,"""bizum""",0,null,null
9843,"""2025-07-28 07:00:00""",99,147,3,"""tarjeta""",0,null,null
9883,"""2025-09-26 13:00:00""",161,70,5,"""tarjeta""",5,"""Álvaro""","""Muñoz Romero"""


In [580]:
for col in ventas_detalle.columns:
    nulos = columna_nula(ventas_detalle, col).height
    print(f"{col}: {nulos} valores nulos")
    

id_venta: 0 valores nulos
fecha: 0 valores nulos
id_cliente: 0 valores nulos
id_producto: 0 valores nulos
cantidad: 0 valores nulos
metodo_pago: 1 valores nulos
descuento: 0 valores nulos
nombre: 130 valores nulos
apellidos: 130 valores nulos


In [581]:
display(id_duplicados(ventas_detalle,"id_venta"))

id_venta,fecha,id_cliente,id_producto,cantidad,metodo_pago,descuento,nombre,apellidos
i64,str,i64,i64,i64,str,i64,str,str
19,"""2025-03-09 19:00:00""",156,117,2,"""tarjeta""",0,null,null
19,"""2025-03-09 19:00:00""",156,117,2,"""tarjeta""",0,null,null


In [582]:
display(columna_nula(ventas_detalle,"metodo_pago"))

id_venta,fecha,id_cliente,id_producto,cantidad,metodo_pago,descuento,nombre,apellidos
i64,str,i64,i64,i64,str,i64,str,str
519,"""2025-06-29 15:00:00""",101,3,1,null,0,null,null


**Ejercicio**

Hacer la transformación para productos y ventas.